In [0]:
from pyspark.sql import functions as F

CATALOG = "dbw_taxrisk_dev"
results, warnings = [], []

def check(name, passed, detail=""):
    results.append((name, bool(passed), str(detail)))

def t(name):
    return spark.table(f"{CATALOG}.{name}")

# ---- Bronze ----
b = t("bronze.posts")
n_b = b.count()
check("bronze.posts no vacía", n_b > 0, n_b)
check("bronze.posts sin datos rescatados (parseo limpio)", b.filter("_rescued_data IS NOT NULL").count() == 0)
check("bronze.posts post_id único", b.select("post_id").distinct().count() == n_b)

# ---- Silver ----
se, sc, cr = t("silver.posts_enriched"), t("silver.posts_classified"), t("silver.creators")
n_se, n_sc = se.count(), sc.count()
check("silver.posts_enriched: una fila por post", se.select("post_id").distinct().count() == n_se, n_se)
check("silver.posts_enriched sin creator_key nulo", se.filter("creator_key IS NULL").count() == 0)
check("silver sin handle en claro", "creator_handle" not in se.columns and "creator_handle" not in sc.columns)
check("silver.posts_classified = posts_enriched", n_sc == n_se, f"{n_sc} vs {n_se}")
allowed = ["paid", "disclosed_collab", "in_kind", "unclear", "organic"]
check("pred_class dentro de los valores permitidos", sc.filter(~F.col("pred_class").isin(allowed)).count() == 0)
check("valor bajo <= medio <= alto",
      sc.filter("value_low > value_mid OR value_mid > value_high").count() == 0)
check("posts con valor tienen valor medio",
      sc.filter("pred_class IN ('in_kind','disclosed_collab','unclear') AND value_mid IS NULL").count() == 0)
check("silver.creators: creator_key único", cr.select("creator_key").distinct().count() == cr.count(), cr.count())

# ---- Gold ----
gf, ge, gr = t("gold.creator_features"), t("gold.creator_exposure"), t("gold.risk_cases")
n_gr = gr.count()
check("creator_features = silver.creators", gf.count() == cr.count(), f"{gf.count()} vs {cr.count()}")
check("suma de n_posts = posts clasificados", gf.agg(F.sum("n_posts")).first()[0] == n_sc)
check("exposición sin valores negativos", ge.filter("tax_exposure_mid < 0 OR expected_value_mid < 0").count() == 0)
check("risk_score entre 0 y 100", gr.filter("risk_score < 0 OR risk_score > 100").count() == 0)
check("ranking 1..N sin repetidos",
      gr.agg(F.min("rank"), F.max("rank"), F.countDistinct("rank")).first()[:3] == (1, n_gr, n_gr))

# ---- Advertencias (no detienen el pipeline) ----
if gr.filter("rule_is_placeholder").count() > 0:
    warnings.append("La exposición usa una regla tributaria placeholder: no es un cálculo validado.")


In [0]:
log = (spark.createDataFrame(results, "check_name string, passed boolean, detail string")
       .withColumn("run_ts", F.current_timestamp()))
log.write.mode("append").saveAsTable(f"{CATALOG}.gold.quality_log")

for name, ok, detail in results:
    print(("PASS" if ok else "FAIL"), "|", name, "|", detail)
for w in warnings:
    print("WARN |", w)

failed = [name for name, ok, _ in results if not ok]
if failed:
    raise Exception(f"{len(failed)} controles de calidad fallaron: " + "; ".join(failed))
print(f"\nTodos los controles pasaron ({len(results)}).")